In [1]:
import numpy as np

def analyze_phase_space_trajectory(signal: np.ndarray, frame_length: int = 512, hop_length: int = 256):
    """
    Maps 1D Audio into a 2D Dynamic Phase Space Trajectory (x(t), v(t)).
    - Voiced: Periodic harmonic loops -> High Orbital Energy + Low Trajectory Chaoticity
    - Unvoiced: Stochastic random walk -> Low Orbital Energy + High Trajectory Chaoticity
    """
    num_frames = 1 + (len(signal) - frame_length) // hop_length
    orbital_energies, trajectory_chaoticity = [], []
    velocity = np.gradient(signal)  # dx/dt (Phase velocity)

    for i in range(num_frames):
        x = signal[i * hop_length : i * hop_length + frame_length]
        v = velocity[i * hop_length : i * hop_length + frame_length]

        # Metric 1: Orbital Energy (2D Phase Radius Squared)
        radius_sq = x**2 + v**2
        orbital_energy = np.mean(radius_sq)

        # Metric 2: Trajectory Chaoticity (Variance of Angular Velocity Progression)
        angles = np.arctan2(v, x)
        angular_velocity = np.diff(np.unwrap(angles))
        chaoticity_variance = np.var(angular_velocity)

        orbital_energies.append(orbital_energy)
        trajectory_chaoticity.append(chaoticity_variance)

    return np.array(orbital_energies), np.array(trajectory_chaoticity)

# Hypothesis Proof Metric
sr = 16000
t = np.linspace(0, 0.1, int(sr * 0.1))
voiced_sig = 0.8 * np.sin(2 * np.pi * 100 * t)       # Voiced sound (100Hz Sine)
unvoiced_sig = 0.15 * np.random.randn(len(t))       # Unvoiced sound (White Noise)

orb_v, cha_v = analyze_phase_space_trajectory(voiced_sig)
orb_u, cha_u = analyze_phase_space_trajectory(unvoiced_sig)

print(f"Voiced   -> Phase Orbital Energy: {orb_v.mean():.4f} | Trajectory Chaoticity: {cha_v.mean():.6f}")
print(f"Unvoiced -> Phase Orbital Energy: {orb_u.mean():.4f} | Trajectory Chaoticity: {cha_u.mean():.6f}")
# Output: Voiced has 10x higher Orbital Energy; Unvoiced has 110x higher Trajectory Chaoticity

Voiced   -> Phase Orbital Energy: 0.3204 | Trajectory Chaoticity: 0.017360
Unvoiced -> Phase Orbital Energy: 0.0350 | Trajectory Chaoticity: 1.881171
